# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

The dataset includes survey data on mental health indicators among residents of Kilifi County, Kenya, with demographic variables and scores from tools such as GAD-7, PHQ-9, and PSQ.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
meta = ds.metadata

print(f"{meta.name}: {meta.description}")
print(f"Published: {meta.datePublished}")
print(f"License: {meta.license}")
print("Keywords:", ', '.join(meta.keywords))

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset may contain multiple record sets, each identified by an `@id`. We'll enumerate available record sets and their fields.

In [ ]:
# Explore record sets and fields using their @id
record_sets = ds.metadata.recordSet      # Should be a list of RecordSet objects
if not record_sets:
    print("No record sets found in the metadata. Please check the schema or contact the data steward.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        print(f"  Description: {rs.get('description', 'N/A')}")
        fields = rs.get('field', [])
        if fields:
            print("  Fields/columns:")
            for f in fields:
                print(f"    Field @id: {f['@id']}")
                print(f"      Name: {f.get('name', 'N/A')}")
                print(f"      DataType: {f.get('dataType', 'N/A')}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we extract and preview all available record sets. All entities are referenced by their `@id`.

In [ ]:
# Extract record set @ids dynamically
record_set_ids = []
if ds.metadata.recordSet:
    record_set_ids = [rs['@id'] for rs in ds.metadata.recordSet]
else:
    print("No record sets to extract.")

# Load each record set as a DataFrame
dfs = {}
for rs_id in record_set_ids:
    records = list(ds.records(record_set=rs_id))   # Each record is a dict
    df = pd.DataFrame(records)
    dfs[rs_id] = df
    print(f"Loaded RecordSet {rs_id}: Shape {df.shape}")

# Preview the first record set (if any)
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Columns for record set {main_rs_id}: {dfs[main_rs_id].columns.tolist()}")
    dfs[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common preprocessing steps: filtering, normalization, grouping, and basic statistics.

We'll operate on fields referenced by their `@id`. Adjust field IDs below as needed for your dataset variables.

In [ ]:
# Example: Select a numeric field for filtering and normalization

# Discover available numeric fields in the first record set
main_rs_id = record_set_ids[0] if record_set_ids else None
df = dfs[main_rs_id] if main_rs_id else pd.DataFrame()

# Try a typical numeric field, such as 'GAD7_score', 'PHQ9_score', or 'PSQ_score' (use their exact @id if available)
numeric_field_id = None
if main_rs_id and ds.metadata.recordSet:
    # Search for likely numeric fields in metadata
    main_rs_meta = next(rs for rs in ds.metadata.recordSet if rs['@id'] == main_rs_id)
    for f in main_rs_meta.get('field', []):
        dt = f.get('dataType', '').lower()
        if 'integer' in dt or 'float' in dt or 'number' in dt:
            numeric_field_id = f['@id']
            break

# Fallback: Use a likely column
if not numeric_field_id:
    for col in df.columns:
        if 'score' in col.lower():
            numeric_field_id = col
            break

# Apply filtering, normalization, and grouping
if main_rs_id and numeric_field_id and numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Example grouping: group by a demographic field (e.g., 'gender', 'level_of_education')
    group_field_id = None
    candidate_group_fields = ['gender', 'Gender', 'level_of_education', 'Level_of_Education', 'marital_status', 'age_group']
    for cg in candidate_group_fields:
        if cg in df.columns:
            group_field_id = cg
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean of {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print("Could not find a suitable numeric field for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we'll create histograms for a numeric mental health score and a simple bar chart for a demographic variable (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15, color="steelblue")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(7,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: numeric field or grouping field not found.")

## 6. Conclusion
This notebook demonstrated how to load, overview, extract, process, and visualize survey data from the Kilifi County FAIR² dataset using `mlcroissant`.

- The dataset contains rich demographic and mental health indicators, ready for analysis with AI tools.
- By referencing fields and record sets using their `@id`, we ensure reproducible and auditable workflows.
- Visual and exploratory analyses can inform local public health and further statistical modeling.

Feel free to further explore additional record sets, fields, and more advanced analyses using the `mlcroissant` library and pandas.